<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Step:
    action: Any
    prob: float  # New: the probability that we ended up here
    reward: float
    state: Any
class MDP:
    def start_state(self) -> Any:
        raise NotImplementedError

    def successors(self, state: Any) -> list[Step]:
        raise NotImplementedError

    def is_end(self, state: Any) -> bool:
        raise NotImplementedError
    def discount(self) -> float:
        raise NotImplementedError
class FlakyTramMDP(MDP):
    def __init__(self, num_locs: int, failure_prob: float):
        self.num_locs = num_locs
        self.failure_prob = failure_prob
    def start_state(self) -> Any:
        return 1

    def successors(self, state: Any) -> list[Step]:
        successors = []
        # Walk
        if state + 1 <= self.num_locs:
            successors.append(Step(action="walk", prob=1, reward=-1, state=state + 1))
        # Tram
        if 2 * state <= self.num_locs:
            # Success: move to desired state
            successors.append(Step(action="tram", prob=1 - self.failure_prob, reward=-2, state=2 * state))
            # Failure: stay in the same state
            successors.append(Step(action="tram", prob=self.failure_prob, reward=-2, state=state))
        return successors
    def is_end(self, state: Any) -> bool:
        return state == self.num_locs
    def discount(self) -> float:
        # No discounting for now
        return 1

#### Environment

In [15]:
import numpy as np

mdp = FlakyTramMDP(num_locs=10, failure_prob=0.4)
np.random.seed(1)

#### Agent

In [16]:
def tram_if_possible_policy(mdp: MDP, state: int) -> str:
    """Need the MDP to know the number of locations to make sure we can take the tram."""
    if state * 2 <= mdp.num_locs:
        return "tram"
    else:
        return "walk"

In [17]:
from typing import Callable
Policy = Callable[[Any], Any]

class RLAlgorithm:
    """
    Abstract class for an RL algorithm, which does two things:
    1. Sends actions to the environment
    2. Incorporates feedback from the environment
    """
    def get_action(self, state: Any) -> Any:
        raise NotImplementedError
    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        raise NotImplementedError

class StaticAgent(RLAlgorithm):
    def __init__(self, policy: Policy):
        self.policy = policy
    def get_action(self, state: Any) -> Any:
        return self.policy(state)
    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        # Setup Reward Feedback Loigic
        pass

In [18]:
from functools import partial

policy = partial(tram_if_possible_policy, mdp)
rl = StaticAgent(policy=policy)
rl.get_action(state=1)

print(f"Action taken: {action}")

Action taken: tram


#### Rollout

In [19]:
def sample_transition(mdp: MDP, state: Any, action: Any) -> Step:
    """
    Samples a transition from the MDP: samples s' with probability T(s, a, s').
    Returns:
    - reward: the reward for the transition
    - next_state: the next state
    - is_end: whether the next state is an end state
    """
    # Get successors given (state, action)
    successors = [successor for successor in mdp.successors(state) if successor.action == action]
    if len(successors) == 0:
        raise ValueError(f"No successors found for state {state} and action {action}")
    # Sample a successor based on its probabilities
    probs = [successor.prob for successor in successors]
    choice = np.random.choice(len(successors), p=probs)
    step = successors[choice]
    return step

In [20]:
def compute_utility(steps: list[Step], discount: float) -> float:
    """Computes the utility (discounted sum of rewards) of a rollout."""
    rewards = [step.reward * discount ** i for i, step in enumerate(steps)]
    utility = sum(rewards)
    return utility

In [21]:
class Rollout:
    """Represents a rollout of an MDP (sequence of actions that produces a utility)."""
    steps: list[Step]
    discount: float
    utility: float  # Discounted sum of rewards
    def __init__(self, steps: list[Step], discount: float):
        self.steps = steps
        self.discount = discount
        self.utility = compute_utility(steps, discount)

In [22]:
def simulate(mdp: MDP, rl: RLAlgorithm, num_trials: int) -> float:
    """
    Runs the RL algorithm on the MDP.
    Return the mean utility of the rollouts.
    """
    utilities = []
    # Repeat multiple times
    for trial in range(num_trials):
        # Environment state
        state = mdp.start_state()
        steps = []
        while not mdp.is_end(state):
            # Agent sends action to environment
            action = rl.get_action(state)
            # Environment sends reward and next state to agent
            step = sample_transition(mdp, state, action)
            rl.incorporate_feedback(state, action, step.reward, step.state, mdp.is_end(step.state))
            steps.append(step)
            state = step.state
        # Compute utility of this rollout
        rollout = Rollout(steps=steps, discount=mdp.discount())
        utilities.append(rollout.utility)
    mean_utility = np.mean(utilities).item()
    return mean_utility

In [23]:
value = simulate(mdp, rl, num_trials=10)
leaderboard = update_leaderboard("tram_if_possible_policy", value)
